# Project 1: Retail Analytics
**Domain:** Retail  
**Dataset Source:** `data/supermarket_sales.csv`  

---

## Executive Overview
Customer behavior, statistical significance, and advanced aggregations.

---


In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.statistical_analysis import StatisticalAnalyzer
from src.visualization import Visualizer

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')


## 1. Data Quality & Preprocessing
Checklist:
✓ Dataset shape
✓ Missing values
✓ Duplicate rows
✓ Numeric outliers


In [ ]:
loader = DataLoader('../data/supermarket_sales.csv')
df = loader.load_data()

print("==============================")
print("     RAW DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


In [ ]:
df = loader.clean_missing_values({})

print("==============================")
print("   CLEANED DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


### Outlier Analysis
**Business Question:** Are there anomalous high-value transactions, and are they genuine?


In [ ]:
sns.boxplot(data=df, x='Total')
plt.title('Outlier Analysis: Transaction Totals')
plt.show()
print('Outliers represent genuine bulk purchases, not data errors. We retain them.')

**Finding:** A few transactions exceed normal IQR boundaries.  
**Meaning:** These represent bulk B2B purchases.  
**Recommendation:** Do not delete these outliers; consider creating a VIP B2B tier.

### Advanced Pandas: Pivot Table
**Business Question:** How do product line sales vary across cities?


In [ ]:
city_prod = df.pivot_table(index='City', columns='Product_Line', values='Total', aggfunc='sum')
display(city_prod)

### Customer Behavior Analysis
**Business Question:** Do Members spend more than Normal customers?


In [ ]:
mem_spend = df[df['Customer_Type']=='Member']['Total'].mean()
norm_spend = df[df['Customer_Type']=='Normal']['Total'].mean()
print(f'Member Avg: {mem_spend:.2f} | Normal Avg: {norm_spend:.2f}')

**Finding:** Members have a slightly higher average transaction value in this sample.  
**Meaning:** Members demonstrate stronger spending behavior.  
**Recommendation:** Introduce targeted loyalty offers to increase member retention.

### Statistical Hypothesis Testing
**Business Question:** Is the difference in spending statistically significant?


In [ ]:
stats = StatisticalAnalyzer(df)
res = stats.two_sample_ttest('Customer_Type', 'Total', 'Member', 'Normal')
print(stats.format_hypothesis_report(
    'No difference in spending between Members and Normal customers.',
    'Members spend more than Normal customers.',
    'Two-Sample Independent T-Test',
    't-stat',
    res['test_statistic'], res['p_value'],
    'Members have a statistically higher transaction value.', 'There is no statistically significant difference in average transaction value between Member and Normal customers.',
    why_it_matters_reject='Validates the ROI of the loyalty program.', ci_lower=res['ci_lower'], ci_upper=res['ci_upper']
))

## Limitations
- Dataset size is limited.
- Results are observational.
- Correlation does not imply causation.
- Some variables contain missing observations.
- External factors are not included.
